In [7]:
# Install Dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Setup: load credentials, build the client.
import os
from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic

# find_dotenv(usecwd=True) walks up from the notebook, so .env is found whether it
# lives in notebooks/ or the repo root. override=True picks up a rotated key.
load_dotenv(find_dotenv(usecwd=True), override=True)

client = Anthropic()          # reads ANTHROPIC_API_KEY from the environment
model = "claude-sonnet-5"

k = os.environ["ANTHROPIC_API_KEY"]
print(f"key {k[:13]}...{k[-4:]} (len={len(k)})  |  model {model}")


key sk-ant-api03-...TwAA (len=108)  |  model claude-sonnet-5


In [9]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


In [10]:
#Make a request
def chat(messages):
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    # response.content is a LIST OF BLOCKS (thinking, text, tool_use, ...).
    # Never assume content[0] is text - branch on block.type.
    return "".join(b.text for b in response.content if b.type == "text")


In [12]:
# Make a Starting list of messages
messages = []

# Add in the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

#Pass the list of messages into 'chat' to get an answer
answer = chat(messages)

#Take the answer and add it to assistant message
add_assistant_message(messages, answer)

#Add in user followup question
add_user_message(messages, "Write another sentence")

answer = chat(messages)

answer

'Instead of using classical bits that are strictly either 0 or 1, quantum computers use **qubits**, which can exist in a superposition of both states simultaneously, enabling them to represent and process a vastly larger number of possibilities at once.'

In [13]:
messages = []

while True:
    # Get user input
    user_input = input("> ")

    # Let the user leave the loop
    if user_input.strip().lower() in {"quit", "exit"}:
        break

    # Add user input to the list of messages
    add_user_message(messages, user_input)

    # Call Claude with the 'chat' function
    answer = chat(messages)

    # Add generated text to the list of messages
    add_assistant_message(messages, answer)

    # Print the generated text
    print("---")
    print(answer)
    print("---")

---
# 1 + 1 = 2

Here's a simple way to visualize it:

🍎 + 🍎 = 🍎🍎

If you have **1 apple** and someone gives you **1 more apple**, you now have **2 apples** in total.

**Basic rule:** When you combine (add) two quantities, you count the total number of items you have altogether.

Would you like to try another addition problem, or see how this connects to bigger concepts like counting or number lines?
---


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: user messages must have non-empty content'}, 'request_id': 'req_011CejXnvr3pLEKMRfoPXLk7'}